# Oracle

This notebook builds the ground-truth oracle

Objectives:
1. Primary: maximize the number of transmission events intercepted.
2. Secondary: minimize total interception delay among schedules
   achieving the maximum possible number of intercepted events.

A transmission event is defined operationally as a continuous run
of 1s in a single frequency band. A 0 breaks the event.

In [1]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

print("NumPy:", np.__version__)

NumPy: 2.2.6


In [2]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# RF environment directory
# ------------------------------------------------------------

TRAIN_RFENV_DIR = Path("./train_RFEnvs")

# Change this later when we want validation.
# For now we are only building/testing the oracle.

# ------------------------------------------------------------
# RF representation
# ------------------------------------------------------------

FREQ_MIN_MHZ = 0
FREQ_MAX_MHZ = 18_000
FREQ_BIN_MHZ = 500

TIME_BIN_S = 0.050

N_FREQ_BINS = int(
    (FREQ_MAX_MHZ - FREQ_MIN_MHZ) / FREQ_BIN_MHZ
)

assert N_FREQ_BINS == 36

print("RF environment directory:", TRAIN_RFENV_DIR.resolve())
print("Frequency bands:", N_FREQ_BINS)
print("Frequency bin:", FREQ_BIN_MHZ, "MHz")
print("Time resolution:", TIME_BIN_S * 1000, "ms")

RF environment directory: C:\Users\vacha\OneDrive\Desktop\Programming\Python\Hackathons\SIH2026 - EW\train_RFEnvs
Frequency bands: 36
Frequency bin: 500 MHz
Time resolution: 50.0 ms


In [3]:
# ============================================================
# 3. LOAD RF ENVIRONMENTS
# ============================================================

rf_files = sorted(
    p for p in TRAIN_RFENV_DIR.glob("*.npy")
    if p.name not in {
        "environment_config.npy",
        "P_baseline.npy",
        "P_baseline_per_environment.npy",
    }
)

if not rf_files:
    raise FileNotFoundError(
        f"No RF environment .npy files found in "
        f"{TRAIN_RFENV_DIR.resolve()}"
    )

print(f"RF environments found: {len(rf_files)}")
print("First few:")
for path in rf_files[:10]:
    print(" ", path.name)

RF environments found: 5
First few:
  config_0.npy
  config_1.npy
  config_10.npy
  config_100.npy
  config_1000.npy


In [4]:
# ============================================================
# 4. TRANSMISSION EVENT EXTRACTION
# ============================================================

def extract_transmission_events(env):
    """
    Extract transmission events from a binary RF environment.

    An event is a continuous run of 1s in ONE frequency band.

    Example:
        0 0 1 1 1 0 1 1

    produces:
        event(start=2, end=4)
        event(start=6, end=7)

    Parameters
    ----------
    env : np.ndarray
        Shape: (time_steps, n_bands)
        Values should be 0/1.

    Returns
    -------
    events : list[dict]
        Each event contains:
            id
            band
            start
            end
            duration
    """

    env = np.asarray(env)

    if env.ndim != 2:
        raise ValueError(
            f"Expected 2-D environment, got shape {env.shape}"
        )

    if env.shape[1] != N_FREQ_BINS:
        raise ValueError(
            f"Expected {N_FREQ_BINS} bands, "
            f"got {env.shape[1]}"
        )

    events = []
    event_id = 0

    n_time_steps = env.shape[0]

    for band in range(N_FREQ_BINS):

        in_event = False
        start = None

        for t in range(n_time_steps):

            active = bool(env[t, band] > 0)

            # Start of a new event
            if active and not in_event:
                start = t
                in_event = True

            # End of an event
            elif not active and in_event:
                end = t - 1

                events.append({
                    "id": event_id,
                    "band": band,
                    "start": start,
                    "end": end,
                    "duration": end - start + 1,
                })

                event_id += 1
                in_event = False

        # Event continues until final timestep
        if in_event:
            end = n_time_steps - 1

            events.append({
                "id": event_id,
                "band": band,
                "start": start,
                "end": end,
                "duration": end - start + 1,
            })

            event_id += 1

    # Sort chronologically, then by band
    events.sort(
        key=lambda e: (e["start"], e["band"])
    )

    # Reassign IDs after sorting
    for new_id, event in enumerate(events):
        event["id"] = new_id

    return events

## Oracle scheduling problem

Each transmission event is:

    event = (band, start, end)

The receiver can scan exactly ONE band per time slot.

An event is intercepted if the oracle scans its band at any
time t satisfying:

    start <= t <= end

Once an event is intercepted, it does not need to be intercepted
again.

For an event:

    delay = (interception_time - start) * TIME_BIN_S

Primary objective:
    maximize number of intercepted events.

Secondary objective:
    among schedules achieving the maximum possible number of
    intercepted events, minimize total interception delay.

The oracle has access to the COMPLETE RF environment, including
future transmission activity. Therefore it is an offline
ground-truth upper bound, not a deployable online scheduler.

In [5]:
# ============================================================
# 8. ORACLE: MIN-COST MAX-FLOW
# ============================================================

class Edge:
    def __init__(self, to, rev, capacity, cost):
        self.to = to
        self.rev = rev
        self.capacity = capacity
        self.cost = cost


class MinCostMaxFlow:
    """
    Simple successive-shortest-augmenting-path
    min-cost max-flow solver.

    Costs are non-negative in our oracle graph.
    """

    def __init__(self, n):
        self.n = n
        self.graph = [[] for _ in range(n)]

    def add_edge(self, u, v, capacity, cost):
        forward = Edge(
            to=v,
            rev=len(self.graph[v]),
            capacity=capacity,
            cost=cost,
        )

        backward = Edge(
            to=u,
            rev=len(self.graph[u]),
            capacity=0,
            cost=-cost,
        )

        self.graph[u].append(forward)
        self.graph[v].append(backward)

    def min_cost_flow(self, source, sink):
        """
        Send as much flow as possible while minimizing total cost.

        Returns
        -------
        flow : int
        cost : int
        """
        flow = 0
        total_cost = 0

        while True:

            dist = [float("inf")] * self.n
            prev_node = [-1] * self.n
            prev_edge = [-1] * self.n

            dist[source] = 0

            # Bellman-Ford style relaxation.
            # Fine for our current oracle implementation and
            # handles the negative reverse edges correctly.
            updated = True

            for _ in range(self.n - 1):

                if not updated:
                    break

                updated = False

                for u in range(self.n):

                    if dist[u] == float("inf"):
                        continue

                    for edge_index, edge in enumerate(
                        self.graph[u]
                    ):

                        if edge.capacity <= 0:
                            continue

                        new_dist = (
                            dist[u] + edge.cost
                        )

                        if new_dist < dist[edge.to]:

                            dist[edge.to] = new_dist
                            prev_node[edge.to] = u
                            prev_edge[edge.to] = edge_index
                            updated = True

            # No more augmenting path
            if dist[sink] == float("inf"):
                break

            # Our capacities are all 1, so one unit per path.
            node = sink

            while node != source:

                u = prev_node[node]
                edge_index = prev_edge[node]

                edge = self.graph[u][edge_index]

                edge.capacity -= 1

                reverse = self.graph[node][edge.rev]
                reverse.capacity += 1

                node = u

            flow += 1
            total_cost += dist[sink]

        return flow, total_cost

In [6]:
# ============================================================
# 9. BUILD ORACLE ASSIGNMENT GRAPH
# ============================================================

def solve_oracle(env):
    """
    Compute the optimal event interception schedule.

    Primary:
        maximize number of intercepted events.

    Secondary:
        minimize total interception delay.

    Returns
    -------
    result : dict
        Contains:
            events
            schedule
            intercepted_events
            missed_events
            max_intercepted_events
            total_delay_steps
            total_delay_s
    """

    env = np.asarray(env, dtype=np.float32)

    if env.ndim != 2:
        raise ValueError(
            f"Expected 2-D environment, got {env.shape}"
        )

    if env.shape[1] != N_FREQ_BINS:
        raise ValueError(
            f"Expected {N_FREQ_BINS} bands, "
            f"got {env.shape[1]}"
        )

    n_time_steps = env.shape[0]

    events = extract_transmission_events(env)

    if not events:

        return {
            "events": [],
            "schedule": np.full(
                n_time_steps,
                -1,
                dtype=int
            ),
            "intercepted_events": [],
            "missed_events": [],
            "max_intercepted_events": 0,
            "total_delay_steps": 0,
            "total_delay_s": 0.0,
        }

    n_events = len(events)

    # --------------------------------------------------------
    # Node layout
    #
    # source
    # event nodes
    # time nodes
    # sink
    # --------------------------------------------------------

    source = 0

    event_offset = 1
    time_offset = event_offset + n_events

    sink = time_offset + n_time_steps

    n_nodes = sink + 1

    flow_solver = MinCostMaxFlow(n_nodes)

    # --------------------------------------------------------
    # Source -> events
    # Each event can be intercepted at most once.
    # --------------------------------------------------------

    for i, event in enumerate(events):

        event_node = event_offset + i

        flow_solver.add_edge(
            source,
            event_node,
            capacity=1,
            cost=0,
        )

    # --------------------------------------------------------
    # Events -> valid interception times
    #
    # Cost = interception delay in time steps.
    #
    # If event starts at t=10 and we intercept at t=13:
    #
    # cost = 13 - 10 = 3
    # --------------------------------------------------------

    for i, event in enumerate(events):

        event_node = event_offset + i

        for t in range(
            event["start"],
            event["end"] + 1
        ):

            time_node = time_offset + t

            delay_steps = t - event["start"]

            flow_solver.add_edge(
                event_node,
                time_node,
                capacity=1,
                cost=delay_steps,
            )

    # --------------------------------------------------------
    # Each time slot can scan only ONE band.
    # Therefore each time node has capacity 1.
    # --------------------------------------------------------

    for t in range(n_time_steps):

        time_node = time_offset + t

        flow_solver.add_edge(
            time_node,
            sink,
            capacity=1,
            cost=0,
        )

    # --------------------------------------------------------
    # Solve
    # --------------------------------------------------------

    max_intercepted, total_delay_steps = (
        flow_solver.min_cost_flow(
            source,
            sink
        )
    )

    # --------------------------------------------------------
    # Recover selected event -> time assignments
    # --------------------------------------------------------

    event_interception_time = {}

    for i, event in enumerate(events):

        event_node = event_offset + i

        for edge in flow_solver.graph[event_node]:

            # Event -> time edge.
            if not (
                time_offset
                <= edge.to
                < time_offset + n_time_steps
            ):
                continue

            # Original capacity was 1.
            # If it is now 0, that edge carries flow.
            if edge.capacity == 0:

                t = edge.to - time_offset

                event_interception_time[
                    event["id"]
                ] = t

                break

    # --------------------------------------------------------
    # Build actual scan schedule.
    #
    # schedule[t] = band scanned at time t
    # schedule[t] = -1 if no event needs to be intercepted
    # --------------------------------------------------------

    schedule = np.full(
        n_time_steps,
        -1,
        dtype=int
    )

    intercepted_events = []

    for event in events:

        event_id = event["id"]

        if event_id not in event_interception_time:
            continue

        t = event_interception_time[event_id]

        schedule[t] = event["band"]

        delay_steps = (
            t - event["start"]
        )

        intercepted_events.append({
            **event,
            "interception_time": t,
            "delay_steps": delay_steps,
            "delay_s": delay_steps * TIME_BIN_S,
        })

    intercepted_ids = {
        e["id"]
        for e in intercepted_events
    }

    missed_events = [
        event
        for event in events
        if event["id"] not in intercepted_ids
    ]

    return {
        "events": events,
        "schedule": schedule,
        "intercepted_events": intercepted_events,
        "missed_events": missed_events,
        "max_intercepted_events": max_intercepted,
        "total_delay_steps": total_delay_steps,
        "total_delay_s": (
            total_delay_steps * TIME_BIN_S
        ),
    }

In [7]:
# ============================================================
# 16. ORACLE METRICS
# ============================================================

def calculate_oracle_metrics(
    oracle_result,
    env
):
    """
    Calculate interpretable metrics from an oracle result.

    These metrics are independent of the bandit implementation.
    """

    events = oracle_result["events"]
    intercepted = oracle_result[
        "intercepted_events"
    ]

    total_events = len(events)
    n_intercepted = len(intercepted)

    if total_events > 0:
        raw_event_interception_rate = (
            n_intercepted / total_events
        )
    else:
        raw_event_interception_rate = np.nan

    if intercepted:
        delays = np.asarray([
            e["delay_s"]
            for e in intercepted
        ])

        mean_delay = float(
            np.mean(delays)
        )

        median_delay = float(
            np.median(delays)
        )

        max_delay = float(
            np.max(delays)
        )

    else:
        mean_delay = np.nan
        median_delay = np.nan
        max_delay = np.nan

    unique_bands_found = len({
        e["band"]
        for e in intercepted
    })

    return {
        "total_events": total_events,
        "intercepted_events": n_intercepted,
        "missed_events": (
            total_events - n_intercepted
        ),
        "raw_event_interception_rate": (
            raw_event_interception_rate
        ),
        "mean_intercept_delay_s": mean_delay,
        "median_intercept_delay_s": median_delay,
        "max_intercept_delay_s": max_delay,
        "unique_bands_found": unique_bands_found,
        "total_delay_s": (
            oracle_result["total_delay_s"]
        ),
    }

In [8]:
# ============================================================
# 18. VERIFY ORACLE SCHEDULE AGAINST GROUND TRUTH
# ============================================================

def verify_oracle_schedule(
    env,
    oracle_result
):
    """
    Independently verify that every claimed oracle
    interception corresponds to a real transmission.

    Also verifies that no event is intercepted more
    than once and that only one band is scanned per timestep.
    """

    schedule = oracle_result["schedule"]
    events = oracle_result["events"]

    # --------------------------------------------------------
    # One scan per time slot
    # --------------------------------------------------------

    assert len(schedule) == len(env)

    # --------------------------------------------------------
    # Check every non-empty scan
    # --------------------------------------------------------

    for t, band in enumerate(schedule):

        if band == -1:
            continue

        assert 0 <= band < N_FREQ_BINS

        # The selected band MUST actually be transmitting.
        assert env[t, band] > 0, (
            f"Invalid oracle scan: "
            f"t={t}, band={band}, "
            f"but environment is 0."
        )

    # --------------------------------------------------------
    # Check event assignments
    # --------------------------------------------------------

    intercepted_ids = set()

    for event in oracle_result[
        "intercepted_events"
    ]:

        event_id = event["id"]

        assert event_id not in intercepted_ids

        intercepted_ids.add(event_id)

        t = event["interception_time"]

        band = event["band"]

        assert (
            event["start"]
            <= t
            <= event["end"]
        )

        assert (
            schedule[t] == band
        )

        assert env[t, band] > 0

    # --------------------------------------------------------
    # Number of schedule entries should equal
    # number of intercepted events.
    # --------------------------------------------------------

    n_scans = np.sum(
        schedule >= 0
    )

    assert (
        n_scans
        == len(
            oracle_result[
                "intercepted_events"
            ]
        )
    )

    print(
        "✓ Oracle schedule independently verified."
    )

In [9]:
# ============================================================
# 20. VISUALIZE ORACLE SCHEDULE
# ============================================================

def plot_oracle_schedule(
    env,
    oracle_result,
    max_time_steps=None
):
    """
    Visualize RF activity and oracle scan decisions.
    """

    if max_time_steps is None:
        max_time_steps = len(env)

    max_time_steps = min(
        max_time_steps,
        len(env)
    )

    plt.figure(figsize=(14, 6))

    # RF activity
    for band in range(N_FREQ_BINS):

        active_times = np.where(
            env[:max_time_steps, band] > 0
        )[0]

        if len(active_times) > 0:

            plt.scatter(
                active_times,
                np.full(
                    len(active_times),
                    band
                ),
                s=8,
                alpha=0.35
            )

    # Oracle scan decisions
    schedule = oracle_result["schedule"]

    scan_times = np.where(
        schedule[:max_time_steps] >= 0
    )[0]

    scan_bands = schedule[
        scan_times
    ]

    plt.scatter(
        scan_times,
        scan_bands,
        marker="x",
        s=45,
        linewidths=2,
        label="Oracle scan"
    )

    plt.xlabel("Time step (50 ms)")
    plt.ylabel("Frequency band")
    plt.title("RF Environment + Oracle Scan Schedule")
    plt.legend()
    plt.grid(True, alpha=0.25)

    plt.show()

In [10]:
# ============================================================
# 22. RUN ORACLE ACROSS ALL TRAINING ENVIRONMENTS
# ============================================================

oracle_results = {}
oracle_metrics = {}

for i, path in enumerate(
    rf_files,
    start=1
):

    env = np.load(
        path
    ).astype(np.float32)

    result = solve_oracle(env)

    metrics = calculate_oracle_metrics(
        result,
        env
    )

    oracle_results[path.name] = result
    oracle_metrics[path.name] = metrics

    print(
        f"[{i:4d}/{len(rf_files)}] "
        f"{path.name:25s} | "
        f"events={metrics['total_events']:5d} | "
        f"intercepted={metrics['intercepted_events']:5d} | "
        f"delay={metrics['total_delay_s']:.3f}s"
    )

[   1/5] config_0.npy              | events=  518 | intercepted=  451 | delay=15.950s
[   2/5] config_1.npy              | events=  279 | intercepted=  269 | delay=2.850s
[   3/5] config_10.npy             | events=  211 | intercepted=  202 | delay=1.450s
[   4/5] config_100.npy            | events=  514 | intercepted=  443 | delay=16.750s
[   5/5] config_1000.npy           | events=   18 | intercepted=   18 | delay=0.000s


In [11]:
# ============================================================
# 23. OVERALL ORACLE SUMMARY
# ============================================================

all_metrics = list(
    oracle_metrics.values()
)

total_events = sum(
    m["total_events"]
    for m in all_metrics
)

total_intercepted = sum(
    m["intercepted_events"]
    for m in all_metrics
)

total_missed = sum(
    m["missed_events"]
    for m in all_metrics
)

total_delay = sum(
    m["total_delay_s"]
    for m in all_metrics
)

mean_delay_per_intercepted = (
    total_delay / total_intercepted
    if total_intercepted > 0
    else np.nan
)

raw_event_rate = (
    total_intercepted / total_events
    if total_events > 0
    else np.nan
)

mean_env_rate = np.nanmean([
    m["raw_event_interception_rate"]
    for m in all_metrics
])

print("=" * 70)
print("ORACLE — ALL TRAINING ENVIRONMENTS")
print("=" * 70)

print(
    f"Environments                    : "
    f"{len(all_metrics)}"
)

print(
    f"Total transmission events       : "
    f"{total_events}"
)

print(
    f"Total events intercepted        : "
    f"{total_intercepted}"
)

print(
    f"Total events missed             : "
    f"{total_missed}"
)

print(
    f"Overall raw event rate          : "
    f"{raw_event_rate:.2%}"
)

print(
    f"Mean per-environment event rate : "
    f"{mean_env_rate:.2%}"
)

print(
    f"Mean delay per intercepted event: "
    f"{mean_delay_per_intercepted:.4f} s"
)

print(
    f"Total oracle delay              : "
    f"{total_delay:.4f} s"
)

ORACLE — ALL TRAINING ENVIRONMENTS
Environments                    : 5
Total transmission events       : 1540
Total events intercepted        : 1383
Total events missed             : 157
Overall raw event rate          : 89.81%
Mean per-environment event rate : 93.08%
Mean delay per intercepted event: 0.0268 s
Total oracle delay              : 37.0000 s


In [12]:
# ============================================================
# 25. SAVE ORACLE RESULTS
# ============================================================

ORACLE_DIR = Path(
    "./Oracle"
)

ORACLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Oracle output directory:",
    ORACLE_DIR.resolve()
)

Oracle output directory: C:\Users\vacha\OneDrive\Desktop\Programming\Python\Hackathons\SIH2026 - EW\Oracle


In [13]:
# ============================================================
# 26. SAVE ORACLE SCHEDULES
# ============================================================

for name, result in oracle_results.items():

    schedule_path = (
        ORACLE_DIR
        / name
        .replace(".npy", "_oracle_schedule.npy")
    )

    np.save(
        schedule_path,
        result["schedule"]
    )

print(
    f"Saved {len(oracle_results)} "
    f"oracle schedules."
)

Saved 5 oracle schedules.


In [14]:
# ============================================================
# 27. SAVE ORACLE METRICS
# ============================================================

import json

metrics_path = (
    ORACLE_DIR
    / "oracle_metrics.json"
)

with open(
    metrics_path,
    "w"
) as f:

    json.dump(
        oracle_metrics,
        f,
        indent=2
    )

print(
    "Saved:",
    metrics_path
)

Saved: Oracle\oracle_metrics.json


In [15]:
# ============================================================
# 28. FINAL ORACLE SANITY CHECKS
# ============================================================

for name, result in oracle_results.items():

    env = np.load(
        TRAIN_RFENV_DIR / name
    ).astype(np.float32)

    # Schedule length
    assert len(
        result["schedule"]
    ) == len(env)

    # At most one scan per time step
    assert np.all(
        (
            result["schedule"] >= -1
        )
        &
        (
            result["schedule"]
            < N_FREQ_BINS
        )
    )

    # Every claimed interception is real
    for event in result[
        "intercepted_events"
    ]:

        t = event["interception_time"]
        band = event["band"]

        assert (
            env[t, band] == 1
        )

        assert (
            event["start"]
            <= t
            <= event["end"]
        )

print(
    "✓ ALL ORACLE SANITY CHECKS PASSED."
)

✓ ALL ORACLE SANITY CHECKS PASSED.


In [16]:
# ============================================================
# 29. VALIDATION ORACLE CONFIGURATION
# ============================================================

VAL_RFENV_DIR = Path("./val_RFEnvs")

VAL_ORACLE_DIR = Path(
    "./val_oracle"
)

VAL_ORACLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Validation RF environment directory:",
    VAL_RFENV_DIR.resolve()
)

print(
    "Validation Oracle output directory:",
    VAL_ORACLE_DIR.resolve()
)

Validation RF environment directory: C:\Users\vacha\OneDrive\Desktop\Programming\Python\Hackathons\SIH2026 - EW\val_RFEnvs
Validation Oracle output directory: C:\Users\vacha\OneDrive\Desktop\Programming\Python\Hackathons\SIH2026 - EW\val_oracle


In [ ]:
# ============================================================
# 30. LOAD VALIDATION RF ENVIRONMENTS
# ============================================================

val_rf_files = sorted(
    p for p in VAL_RFENV_DIR.glob("*.npy")
    if p.name not in {
        "environment_config.npy",
        "P_baseline.npy",
        "P_baseline_per_environment.npy",
    }
)

if not val_rf_files:

    raise FileNotFoundError(
        f"No RF environment .npy files found in "
        f"{VAL_RFENV_DIR.resolve()}"
    )

print(
    f"Validation RF environments found: "
    f"{len(val_rf_files)}"
)

print("Validation files:")

for path in val_rf_files:
    print(" ", path.name)

Validation RF environments found: 4
Validation files:
  config_0.npy
  config_1.npy
  config_10.npy
  config_100.npy


In [18]:
# ============================================================
# 31. RUN ORACLE ACROSS ALL VALIDATION ENVIRONMENTS
# ============================================================

val_oracle_results = {}
val_oracle_metrics = {}

for i, path in enumerate(
    val_rf_files,
    start=1
):

    env = np.load(
        path
    ).astype(np.float32)

    result = solve_oracle(
        env
    )

    metrics = calculate_oracle_metrics(
        result,
        env
    )

    val_oracle_results[
        path.name
    ] = result

    val_oracle_metrics[
        path.name
    ] = metrics

    print(
        f"[{i:4d}/{len(val_rf_files)}] "
        f"{path.name:25s} | "
        f"steps={len(env):5d} | "
        f"events={metrics['total_events']:5d} | "
        f"intercepted={metrics['intercepted_events']:5d} | "
        f"ISR={metrics['raw_event_interception_rate']:.2%} | "
        f"delay={metrics['total_delay_s']:.3f}s"
    )

[   1/4] config_0.npy              | steps=  535 | events=  220 | intercepted=  208 | ISR=94.55% | delay=7.500s
[   2/4] config_1.npy              | steps=  850 | events=  544 | intercepted=  489 | ISR=89.89% | delay=16.250s
[   3/4] config_10.npy             | steps=  598 | events=  367 | intercepted=  312 | ISR=85.01% | delay=7.900s
[   4/4] config_100.npy            | steps=  593 | events=  393 | intercepted=  342 | ISR=87.02% | delay=12.500s


In [19]:
# ============================================================
# 32. VERIFY VALIDATION ORACLE SCHEDULES
# ============================================================

for name, result in val_oracle_results.items():

    env = np.load(
        VAL_RFENV_DIR / name
    ).astype(np.float32)

    print(
        f"\nChecking {name}..."
    )

    verify_oracle_schedule(
        env,
        result
    )

print(
    "\n✓ ALL VALIDATION ORACLE SCHEDULES VERIFIED."
)


Checking config_0.npy...
✓ Oracle schedule independently verified.

Checking config_1.npy...
✓ Oracle schedule independently verified.

Checking config_10.npy...
✓ Oracle schedule independently verified.

Checking config_100.npy...
✓ Oracle schedule independently verified.

✓ ALL VALIDATION ORACLE SCHEDULES VERIFIED.


In [20]:
# ============================================================
# 33. SAVE VALIDATION ORACLE SCHEDULES
# ============================================================

for name, result in val_oracle_results.items():

    schedule_path = (
        VAL_ORACLE_DIR
        / name.replace(
            ".npy",
            "_oracle_schedule.npy"
        )
    )

    np.save(
        schedule_path,
        result["schedule"]
    )

    print(
        "Saved:",
        schedule_path
    )

print(
    f"\n✓ Saved {len(val_oracle_results)} "
    f"validation Oracle schedules."
)

Saved: val_oracle\config_0_oracle_schedule.npy
Saved: val_oracle\config_1_oracle_schedule.npy
Saved: val_oracle\config_10_oracle_schedule.npy
Saved: val_oracle\config_100_oracle_schedule.npy

✓ Saved 4 validation Oracle schedules.


In [21]:
# ============================================================
# 34. SAVE VALIDATION ORACLE METRICS
# ============================================================

val_metrics_path = (
    VAL_ORACLE_DIR
    / "val_oracle_metrics.json"
)

with open(
    val_metrics_path,
    "w"
) as f:

    json.dump(
        val_oracle_metrics,
        f,
        indent=2
    )

print(
    "Saved:",
    val_metrics_path
)

Saved: val_oracle\val_oracle_metrics.json


In [22]:
# ============================================================
# 35. VALIDATION ORACLE SUMMARY
# ============================================================

val_all_metrics = list(
    val_oracle_metrics.values()
)

val_total_events = sum(
    m["total_events"]
    for m in val_all_metrics
)

val_total_intercepted = sum(
    m["intercepted_events"]
    for m in val_all_metrics
)

val_total_delay = sum(
    m["total_delay_s"]
    for m in val_all_metrics
)

val_micro_isr = (
    val_total_intercepted
    / val_total_events
    if val_total_events > 0
    else np.nan
)

val_mean_delay = (
    val_total_delay
    / val_total_intercepted
    if val_total_intercepted > 0
    else np.nan
)

print("=" * 70)
print("ORACLE — ALL VALIDATION ENVIRONMENTS")
print("=" * 70)

print(
    f"Environments             : "
    f"{len(val_all_metrics)}"
)

print(
    f"Total transmission events: "
    f"{val_total_events}"
)

print(
    f"Events intercepted       : "
    f"{val_total_intercepted}"
)

print(
    f"Events missed            : "
    f"{val_total_events - val_total_intercepted}"
)

print(
    f"Oracle micro ISR         : "
    f"{val_micro_isr:.2%}"
)

print(
    f"Mean interception delay  : "
    f"{val_mean_delay:.4f} s"
)

ORACLE — ALL VALIDATION ENVIRONMENTS
Environments             : 4
Total transmission events: 1524
Events intercepted       : 1351
Events missed            : 173
Oracle micro ISR         : 88.65%
Mean interception delay  : 0.0327 s
